In [1]:
from langchain_experimental.text_splitter import SemanticChunker

C:\Users\PC\AppData\Local\Temp\ipykernel_13108\2829801429.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [1]:
#pip install langchain-experimental
from langchain_community.document_loaders import PyPDFLoader
docs=PyPDFLoader(r"C:\Users\PC\OneDrive\Documents\agentic ai.pdf").load()
print(len(docs))

603


In [3]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddin=HuggingFaceEmbeddings(model_name='all-miniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
chunker=SemanticChunker(
    embeddings=HuggingFaceEmbeddings(),
    breakpoint_threshold_type='percentile',
    breakpoint_threshold_amount=95,
)
chnks=chunker.split_documents(docs)

C:\Users\PC\AppData\Local\Temp\ipykernel_21620\3725318113.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(),
C:\Users\PC\AppData\Local\Temp\ipykernel_21620\3725318113.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings=HuggingFaceEmbeddings(),


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [29]:
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks=RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=32).split_documents(docs)
#vectorstore=Chroma.from_documents(chunks,embeddin)


# multi query genration

In [ ]:
%pip uninstall -y lasngchain
%pip install langchain==0.3.27

Found existing installation: langchain 1.3.14
Uninstalling langchain-1.3.14:
  Successfully uninstalled langchain-1.3.14
Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain.retrievers import MultiQueryRetriever
vectorstore=Chroma.from_documents(chunks,embedding)

                                 

In [38]:
#from langchain_ollama import ChatOllama
#!pip install -U langchain-core langchain-ollama

In [1]:
#!pip install --upgrade --force-reinstall langchain-core langchain-ollama


In [37]:
from langchain.retrievers import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
retrievers=vectorstore.as_retriever()
from langchain_ollama import ChatOllama
llm=ChatOllama(model='smollm:135m')
compressor=LLMLinguaCompressor.from_llm(llm)
retriver=ContextualCompressionRetriever.from_llm(llm=llm,retriever=retrievers,include_original=True)

AttributeError: from_llm

In [31]:
#!pip install --upgrade "langchain-core<1.0.0" "langchain-ollama<1.0.0" "langchain<1.0.0"

In [34]:
query='what is RAG ?'

In [2]:
#result=retriver.invoke(query)
#for i ,doc in enumerate(result):
 #   print(f"...result{i+1}==")
  #  print(doc.page_content)
   # print('*'*20)
#print(result)
# I MUTE THIS BE

In [39]:
from langchain.retrievers.document_compressors import LLMChainExtractor
compresor=LLMChainExtractor.from_llm(llm)
compreesoin_retriver=ContextualCompressionRetriever(base_retriever=retrievers,base_compressor=compresor)

In [2]:
#q='what is RAG?'
#res=compreesoin_retriver.invoke(q)
#for i,doc in enumerate(res):
 #   print(f'result{i+1}=')
  #  print(doc.page_content)
   # print('*'*50)

In [44]:
docs1=retriver.get_relevant_documents(query)
#print(docs1)
result=retriver.invoke(query)
for i ,doc in enumerate(docs1):
    print(f"...result{i+1}==")
    print(doc.page_content)
    print('*'*20)

...result1==
• Query expansion:generate multiple paraphrases of the query and take the union of retrieved
results.
• Step-back prompting:abstract the specific query to a more general question before retrieval.
325
********************
...result2==
intermediate results. The implementation below uses LangGraph to wire four nodes into a loop:
1.Plan: Decompose the user query into sub-queries (one per information need).
2.Retrieve: Route each sub-query to the appropriate source and fetch documents.
3.Evaluate: Judge whether the accumulated context is sufficient to answer the original query.
********************
...result3==
multiple focused sub-queries, each designed to direct the model’s attention to a specific part of the
context:
1. Query decomposition: Break the user question into atomic sub-questions that each target a
narrow aspect.
2. Attentive retrieval: For each sub-query, retrieve or highlight only the relevant context
slice—forcing the model to attend to it.
********************

In [49]:
print(docs1[0].page_content)

• Query expansion:generate multiple paraphrases of the query and take the union of retrieved
results.
• Step-back prompting:abstract the specific query to a more general question before retrieval.
325


# reranking

In [2]:
#!pip install sentence-transformers

In [ ]:
#from sentence_transformers import CrossEncoder
#reeanker=CrossEncoder("BAAI/bge-reranker-large")
#def rerank(query:str,docs:list[str],top_n:int=5)-> list[str]:
 #   pairs=[(query,doc) for doc in docs]
  #  scored=rerank.predict(pairs)
   # ranked=sorted(zip(scored,docs),reverse=True)
    #return [doc for _,doc in ranked[:top_n]]
    

In [ ]:
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder

retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

docs = retriever.invoke(query)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [[query, doc.page_content] for doc in docs]
scores = reranker.predict(pairs)

docs = [
    doc for _, doc in sorted(
        zip(scores, docs),
        reverse=True,
        key=lambda x: x[0]
    )
]

# Top 5 documents
docs = docs[:5]

# segamntic chunking


In [12]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
chunker=SemanticChunker(embeddings=HuggingFaceEmbeddings(),breakpoint_threshold_type='percentile',
                       breakpoint_threshold_amount=95,)



C:\Users\PC\AppData\Local\Temp\ipykernel_13108\2596775244.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  chunker=SemanticChunker(embeddings=HuggingFaceEmbeddings(),breakpoint_threshold_type='percentile',


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
from langchain_community.document_loaders import  PyPDFLoader
docs=PyPDFLoader(r"C:\Users\PC\Downloads\Microsoft 2025 Annual Report.pdf").load()
print(len(docs))

84


In [ ]:
chunks=chunker.split_documents(docs)

In [1]:
from langchain_community.vectorstores import Chroma
#vectorestore=Chroma.from_documents(documents=chunks,embedding=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2'))

C:\Users\PC\AppData\Local\Temp\ipykernel_12832\3140620758.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [17]:
bad_chunks = [
    (i, type(doc.page_content), doc.page_content)
    for i, doc in enumerate(chunks)
    if not isinstance(doc.page_content, str)
]

print(bad_chunks[:5])

[]
